# 15 — lmfit superpixel inversion with the bi_jax water model

Demonstrates the two-layer architecture:

| Layer | Module | Role |
|---|---|---|
| Layer 1 — solver | `lmfit_engine` | per-spectrum Levenberg–Marquardt LSQ |
| Layer 2 — image  | `superpixel_engine` | SLIC segmentation + PCA/kNN back-interpolation |

Forward model: **bi_jax** (Bi et al. 2023, HEREON water optical model).

Free parameters: `C_0` (phytoplankton), `C_Y` (CDOM), `C_ism` (ISM), `offset`.

## 1  Imports & configuration

In [ ]:
import numpy as np
import lmfit
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from bio_optics.water.reflectance import bi_jax
from bio_optics.inversion import lmfit_engine
from bio_optics.image_processing import superpixel_engine

In [ ]:
# --- inversion config --------------------------------------------------------
NOISE       = 0.001    # Rrs noise level [sr-1]
N_SEGMENTS  = 1000     # target SLIC superpixel count
COMPACTNESS = 0.1      # SLIC: low = spectral boundaries dominate
SLIC_SIGMA  = 2.0      # SLIC: Gaussian smoothing
K           = 4        # kNN neighbours for back-interpolation
N_COMPONENTS = 6       # PCA dimensions for spectral embedding
MAX_NFEV    = 400      # max lmfit function evaluations per spectrum

## 2  Load image data

Adapt the cell below to your data source.  
`Rrs_arr` must be `(n_rows, n_cols, n_obs)` in sr⁻¹; `wavelengths` must be `(n_obs,)` in nm.

In [ ]:
# --- replace this block with your own data loading --------------------------
import xarray as xr

img = xr.open_zarr("<path-to-your-enmap-scene.zarr>")   # adjust path
wavelengths = img.wavelength.values[:80]                  # first 80 EnMAP bands

refl     = img['refl'].values[:, :, :80].astype(np.float64)
Rrs_arr  = refl / np.pi / 10_000                          # reflectance → Rrs [sr-1]

# mask land / cloud (set to NaN; superpixel engine propagates NaN correctly)
water_mask = img['water_mask'].values.astype(bool)        # True = water
Rrs_arr[~water_mask] = np.nan

print(f"Image shape : {Rrs_arr.shape}")
print(f"Wavelengths : {wavelengths[0]:.1f} – {wavelengths[-1]:.1f} nm ({len(wavelengths)} bands)")

## 3  Precompute spectral tables

In [ ]:
pre = bi_jax.precompute(wavelengths)
print(f"n_phy_classes : {pre['n_classes']}")

## 4  Define lmfit parameters

Free parameters: `C_0`, `C_Y`, `C_ism`, `offset`.  
Everything else is fixed to literature defaults.

In [ ]:
params = lmfit.Parameters()

# --- free parameters --------------------------------------------------------
params.add('C_0',   value=2.0,  min=0.0,   max=200.0, vary=True)   # phy class 0 [ug/L]
params.add('C_Y',   value=0.2,  min=0.0,   max=10.0,  vary=True)   # CDOM abs at lambda_0_cdom [1/m]
params.add('C_ism', value=2.0,  min=0.0,   max=200.0, vary=True)   # ISM [g/m3]
params.add('offset',value=0.0,  min=-0.02, max=0.02,  vary=True)   # spectral offset [sr-1]

# --- phytoplankton classes 1–7 (fixed to 0) ---------------------------------
for i in range(1, 8):
    params.add(f'C_{i}', value=0.0, vary=False)

# --- CDOM -------------------------------------------------------------------
params.add('S_cdom',        value=0.014, vary=False)   # spectral slope [1/nm]
params.add('lambda_0_cdom', value=440.0, vary=False)   # reference wavelength [nm]
params.add('K',             value=0.0,   vary=False)   # baseline absorption [1/m]

# --- minerogenic detritus absorption ----------------------------------------
params.add('A_md',        value=0.04,  vary=False)
params.add('S_md',        value=0.011, vary=False)
params.add('C_md',        value=0.0,   vary=False)
params.add('lambda_0_md', value=440.0, vary=False)

# --- biogenic detritus absorption -------------------------------------------
params.add('A_bd',        value=0.001, vary=False)
params.add('S_bd',        value=0.011, vary=False)
params.add('C_bd',        value=0.0,   vary=False)
params.add('lambda_0_bd', value=440.0, vary=False)

# --- detritus attenuation ---------------------------------------------------
params.add('gamma_d',     value=0.5,   vary=False)
params.add('x0',          value=0.96,  vary=False)
params.add('x1',          value=0.5,   vary=False)
params.add('x2',          value=1.0,   vary=False)
params.add('lambda_0_c_d',value=550.0, vary=False)

# --- temperature ------------------------------------------------------------
params.add('T_W',   value=15.0, vary=False)   # water temperature [°C]
params.add('T_W_0', value=15.0, vary=False)   # reference temperature [°C]

# --- phytoplankton packaging ------------------------------------------------
params.add('A_phy',        value=0.06,  vary=False)
params.add('E0',           value=0.65,  vary=False)
params.add('E1',           value=0.67,  vary=False)
params.add('lambda_0_phy', value=440.0, vary=False)

# --- backscattering ratios --------------------------------------------------
for i in range(8):
    params.add(f'b_ratio_C_{i}', value=0.01, vary=False)
params.add('b_ratio_md', value=0.02, vary=False)
params.add('b_ratio_bd', value=0.01, vary=False)

# --- Lee et al. (2011) coefficients ----------------------------------------
params.add('Gw0', value=0.0895, vary=False)
params.add('Gw1', value=0.1247, vary=False)
params.add('Gp0', value=0.0401, vary=False)
params.add('Gp1', value=0.0841, vary=False)

free = [n for n, p in params.items() if p.vary]
print(f"Free parameters ({len(free)}): {free}")

## 5  Forward function adapter

`bi_jax.forward(params, precomputed)` embeds wavelengths inside `precomputed`, so we wrap it
to match the `lmfit_engine` signature `forward_func(params, wavelengths)`.

In [ ]:
def forward_func(params, wavelengths):
    """Adapter: lmfit_engine signature → bi_jax.forward."""
    return np.array(bi_jax.forward(params, pre))

# quick sanity check
Rrs_test = forward_func(params, wavelengths)
print(f"Forward check — Rrs range: [{Rrs_test.min():.4f}, {Rrs_test.max():.4f}] sr-1")

plt.figure(figsize=(8, 3))
plt.plot(wavelengths, Rrs_test)
plt.xlabel('Wavelength [nm]')
plt.ylabel('Rrs [sr⁻¹]')
plt.title('bi_jax forward — default parameters')
plt.tight_layout()

## 6  Build LmfitSetup

In [ ]:
setup = lmfit_engine.build_inversion(
    params,
    wavelengths,
    forward_func,
    method='least-squares',
    max_nfev=MAX_NFEV,
)
print(f"fit_names : {setup.fit_names}")

## 7  Superpixel inversion

In [ ]:
results = superpixel_engine.invert_image_superpixel(
    Rrs_arr,
    setup,
    NOISE,
    invert_fn=lmfit_engine.invert_image,
    n_segments=N_SEGMENTS,
    compactness=COMPACTNESS,
    sigma=SLIC_SIGMA,
    k=K,
    n_components=N_COMPONENTS,
    store_sp_results=True,
)

x_hat      = results['x_hat']           # (n_rows, n_cols, n_fit)
chi2       = results['chi2']            # (n_rows, n_cols)
chi2_cal   = results['chi2_calibrated'] # chi2 × sp_counts
fit_names  = results['fit_names']
labels     = results['labels']
sp_counts  = results['sp_counts']

print(f"fit_names      : {fit_names}")
print(f"x_hat shape    : {x_hat.shape}")
print(f"n_superpixels  : {len(np.unique(labels))}")
print(f"median sp size : {np.median(sp_counts):.0f} px")

## 8  Results

In [ ]:
# --- per-parameter maps -----------------------------------------------------
param_labels = {
    'C_0':    ('Phytoplankton C₀', 'µg/L',  'YlGn',  0, 50),
    'C_Y':    ('CDOM C_Y',         '1/m',   'YlOrBr', 0, 3),
    'C_ism':  ('ISM',              'g/m³',  'Greys',  0, 50),
    'offset': ('Spectral offset',  'sr⁻¹',  'RdBu',  -0.005, 0.005),
}

n_params = len(fit_names)
fig, axes = plt.subplots(1, n_params, figsize=(4 * n_params, 4))

for ax, name in zip(axes, fit_names):
    idx   = fit_names.index(name)
    label, unit, cmap, vmin, vmax = param_labels.get(
        name, (name, '', 'viridis', None, None)
    )
    im = ax.imshow(x_hat[..., idx], cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label=unit, shrink=0.8)
    ax.set_title(label)
    ax.axis('off')

plt.suptitle('bi_jax superpixel inversion — retrieved parameters', fontsize=12)
plt.tight_layout()

In [ ]:
# --- superpixel segment map + chi2 ------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(labels, cmap='tab20b')
axes[0].set_title(f'SLIC segments (n={len(np.unique(labels))})')
axes[0].axis('off')

im1 = axes[1].imshow(chi2, cmap='plasma', vmin=0, vmax=np.nanpercentile(chi2, 95))
plt.colorbar(im1, ax=axes[1], shrink=0.8)
axes[1].set_title('chi² (at pixel noise)')
axes[1].axis('off')

im2 = axes[2].imshow(chi2_cal, cmap='plasma', vmin=0, vmax=np.nanpercentile(chi2_cal, 95))
plt.colorbar(im2, ax=axes[2], shrink=0.8)
axes[2].set_title('chi² calibrated (× sp_counts)')
axes[2].axis('off')

plt.tight_layout()

In [ ]:
# --- spot-check: measured vs fitted spectrum for a random superpixel --------
sp_res    = results['sp_results']
sp_spectra = results['sp_results'].get('sp_spectra') if 'sp_spectra' in results.get('sp_results', {}) else None

# pick the 5 largest superpixels for the check
top5_idx = np.argsort(sp_counts)[-5:]

fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=True)
for ax, idx in zip(axes, top5_idx):
    # back out superpixel mean spectra from aggregate_superpixels output
    sp_x = sp_res['x_hat'][idx]      # (n_fit,) physical retrieved values
    # reconstruct fitted params for this superpixel
    p_fit = params.copy()
    for name, val in zip(fit_names, sp_x):
        p_fit[name].value = val
    Rrs_fit = forward_func(p_fit, wavelengths)
    ax.plot(wavelengths, Rrs_fit, label='fit')
    ax.set_title(f'SP {idx} (n={sp_counts[idx]})')
    ax.set_xlabel('λ [nm]')
    ax.set_ylabel('Rrs [sr⁻¹]') if ax == axes[0] else None

plt.suptitle('Fitted spectra — 5 largest superpixels', fontsize=11)
plt.tight_layout()